# DD-PRiSM-plus — Build the L1000 drug signatures

Produces `drug_signatures_l1000.parquet`: one 978-landmark-gene transcriptomic
signature per drug, from **LINCS Phase I (GSE92742)**, mapped to NSC by
InChIKey. **CPU session** — no GPU needed.

Attach two inputs via **Add Input**:

1. Your **01_setup_and_data** output — for `nsc_smiles.csv`.
2. A **Kaggle Dataset holding the GSE92742 Level-5 files** (below).

The GSE92742 files, from the [GEO page](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE92742)
(uncompress the `.gctx.gz` first):

| file | size |
|---|---|
| `GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx` | ~23 GB |
| `GSE92742_Broad_LINCS_gene_info.txt` | small |
| `GSE92742_Broad_LINCS_sig_info.txt` | ~100 MB |
| `GSE92742_Broad_LINCS_pert_info.txt` | small |

> The `.gctx` is big, so attach it as a **Dataset** — it then lives in
> read-only `/kaggle/input` and does **not** count against your 20 GB working
> disk. Only the 978 landmark rows and the signatures of the drugs this study
> touches are ever read, so memory stays in the hundreds of MB.

In [ ]:
import os, glob, subprocess

REPO = '/kaggle/working/ddprism-plus'
OUT  = '/kaggle/working/l1000'

ok = subprocess.run(['curl', '-sI', '--max-time', '15', 'https://github.com'],
                    capture_output=True).returncode == 0
print('internet:', 'ON' if ok else 'OFF  <-- enable it, then rerun')

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}
os.chdir(REPO)
print('working in', os.getcwd())

## Install

`cmapPy` reads the `.gctx`; `rdkit` canonicalises SMILES to InChIKeys for the
NSC↔L1000 mapping.

In [ ]:
!pip install --quiet cmapPy rdkit pyarrow
print('installed')

## Find the inputs

The four GSE92742 files and `nsc_smiles.csv` are located wherever Kaggle
mounted them, then linked into one staging folder so the builder sees them
together.

In [ ]:
import glob, os

def one(pattern, what):
    hits = glob.glob('/kaggle/input/**/' + pattern, recursive=True)
    if not hits:
        raise SystemExit('missing ' + what + ' (' + pattern + ') -- attach the '
                         'dataset that holds it via Add Input.')
    return hits[0]

gctx     = one('*Level5_COMPZ.MODZ*.gctx', 'the Level-5 MODZ matrix')
geneinfo = one('*gene_info*.txt',          'gene_info')
siginfo  = one('*sig_info*.txt',           'sig_info')
pertinfo = one('*pert_info*.txt',          'pert_info')
smiles   = one('nsc_smiles.csv',           'nsc_smiles.csv (attach 01 output)')

LINCS = '/kaggle/working/lincs'
DATA  = '/kaggle/working/raw'
os.makedirs(LINCS, exist_ok=True)
os.makedirs(DATA, exist_ok=True)
for src in (gctx, geneinfo, siginfo, pertinfo):
    dst = os.path.join(LINCS, os.path.basename(src))
    if not os.path.exists(dst):
        os.symlink(src, dst)
sm = os.path.join(DATA, 'nsc_smiles.csv')
if not os.path.exists(sm):
    os.symlink(smiles, sm)

print('gctx     :', gctx)
print('sig_info :', siginfo)
print('nsc_smiles:', smiles)

## Build the signatures

Reads only the 978 landmark rows and the signatures of drugs it can map to an
NSC, averages each drug's signatures into one consensus vector, and writes the
parquet. `--all-compounds` maps every NSC with a SMILES; to restrict to just
the drugs the study uses, attach the `02_preprocess` output and point `--out`
at it instead.

**Read the coverage line it prints** — it says how many of the drugs got an
L1000 signature. That is the honest scope of the whole L1000 experiment.

In [ ]:
!python scripts/fetch_l1000.py --lincs {LINCS} --data {DATA} --out {OUT} --all-compounds

## What came out

In [ ]:
import pandas as pd, os
frame = pd.read_parquet(os.path.join(OUT, 'drug_signatures_l1000.parquet'))
print('shape:', frame.shape)
print('drugs:', f'{len(frame):,}')
frame.iloc[:3, :6]

## Save it

**Save Version → Save & Run All (Commit).** The `08_experiment_l1000` and
`09_experiment_l1000_only` notebooks attach this output alongside the
`02_preprocess` output, via **Add Input → Your Work → Notebook Output**.